# Phase 3: Matrix Factorization (SVD)

In this notebook, we implement Singular Value Decomposition (SVD), a matrix factorization algorithm. SVD handles data sparsity well by factoring the user-item matrix into dense latent factors. We use the `scikit-surprise` library for this implementation.

Steps:
1. Load the SVD subset.
2. Create an 80/20 train/test split.
3. Train the SVD model.
4. Predict ratings and calculate RMSE.
5. Generate personalized recommendations for users.


In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')


## 1. Load Data
We load our dedicated SVD subset (approx. 6.9 million ratings) and the movie titles.

In [2]:
# Load the SVD-specific subset
df = pd.read_csv('../data/processed/netflix_svd_subset.csv')

# Load movie titles
titles_df = pd.read_csv('../data/raw/movie_titles.csv', encoding='ISO-8859-1', header=None, names=['movie_id', 'year', 'title'], on_bad_lines='skip')
title_map = dict(zip(titles_df['movie_id'], titles_df['title']))

print(f"Loaded {len(df):,} ratings.")


Loaded 6,924,035 ratings.


## 2. Train/Test Split
Surprise requires a `Reader` to parse the dataframe. We specify a 1-5 rating scale and split the data 80% for training and 20% for testing.

In [3]:
# Parse data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'movie_id', 'rating']], reader)

# Create train/test split
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f"Training set: {trainset.n_ratings:,} ratings")
print(f"Test set: {len(testset):,} ratings")


Training set: 5,539,228 ratings
Test set: 1,384,807 ratings


## 3. Train the SVD Model
We initialize the SVD model with 50 latent factors and 20 epochs to keep training efficient while maintaining reasonable accuracy.

In [4]:
# Initialize SVD with requested parameters
model = SVD(n_factors=50, n_epochs=20, random_state=42)

# Train the model
print("Training SVD model...")
model.fit(trainset)
print("Training complete!")


Training SVD model...


Training complete!


## 4. Evaluate Model (RMSE)
We calculate the Root Mean Squared Error (RMSE) on the test set to evaluate baseline predictive accuracy.

In [5]:
# Predict ratings for the test set
predictions = model.test(testset)

# Compute RMSE
rmse = accuracy.rmse(predictions)
print(f"SVD RMSE: {rmse:.4f}")


RMSE: 0.8013
SVD RMSE: 0.8013


## 5. Prediction and Recommendation Functions
These functions allow us to predict a specific rating and generate a list of top-N recommendations for any user.

In [6]:
def predict_rating(user_id, movie_id):
    """Predicts the rating a user would give to a specific movie."""
    pred = model.predict(uid=user_id, iid=movie_id)
    return pred.est

def recommend_for_user_svd(user_id, top_n=10):
    """Generates Top-N movie recommendations for a user."""
    all_movies = df['movie_id'].unique()
    user_rated_movies = df[df['user_id'] == user_id]['movie_id'].values
    
    # Isolate unrated movies
    unrated_movies = np.setdiff1d(all_movies, user_rated_movies)
    
    # Predict rating for each unrated movie
    predictions = []
    for m_id in unrated_movies:
        est = predict_rating(user_id, m_id)
        predictions.append((m_id, est))
        
    # Sort by highest predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_predictions = predictions[:top_n]
    
    # Format output
    results = []
    for m_id, est in top_predictions:
        title = title_map.get(m_id, f"Unknown (ID: {m_id})")
        results.append({"movie_id": m_id, "title": title, "predicted_rating": round(est, 2)})
        
    return pd.DataFrame(results)


## 6. Demonstrate Recommendations
Let's generate personalized recommendations for 5 sample users.

In [7]:
# Test with 5 sample users
sample_users = df['user_id'].unique()[:5]

for u in sample_users:
    print(f"--- Top 5 SVD Recommendations for User: {u} ---")
    display(recommend_for_user_svd(u, top_n=5))
    print()


--- Top 5 SVD Recommendations for User: 2031561 ---


,movie_id,title,predicted_rating
0,1476,Six Feet Under: Season 4,4.50
1,3446,Spirited Away,4.48
2,68,Invader Zim,4.43
3,8637,The Corporation,4.43
4,7569,Dead Like Me: Season 2,4.38



--- Top 5 SVD Recommendations for User: 701730 ---


,movie_id,title,predicted_rating
0,7742,Six Feet Under: Season 3,4.69
1,12398,Veronica Mars: Season 1,4.61
2,1476,Six Feet Under: Season 4,4.54
3,12125,The Blue Planet: Seas of Life: Open Ocean - Th...,4.54
4,1595,The Third Man,4.54



--- Top 5 SVD Recommendations for User: 1614320 ---


,movie_id,title,predicted_rating
0,11662,The Sopranos: Season 5,3.99
1,1915,Law & Order: Special Victims Unit: The Second ...,3.98
2,17146,Doctor Zhivago,3.97
3,5258,Ken Burns' Civil War,3.96
4,11050,MASH: Season 6,3.96



--- Top 5 SVD Recommendations for User: 52540 ---


,movie_id,title,predicted_rating
0,1561,American Wedding,5.0
1,4078,Larry the Cable Guy: Git-R-Done,5.0
2,4561,Everybody Loves Raymond: Season 4,5.0
3,5239,The Longest Yard,5.0
4,10883,The Titanic,5.0



--- Top 5 SVD Recommendations for User: 1198785 ---


,movie_id,title,predicted_rating
0,7569,Dead Like Me: Season 2,4.74
1,3662,House of Cards Trilogy II: To Play the King,4.53
2,12398,Veronica Mars: Season 1,4.53
3,10406,Absolutely Fabulous: Series 3,4.48
4,2585,Absolutely Fabulous: Series 2,4.43


## 7. Discussion

**Strengths of SVD:**
- **Accuracy:** By optimizing latent factors to minimize global error, SVD generally achieves a lower RMSE than neighborhood models.
- **Handles Sparsity:** The algorithm projects users and items into a dense latent space, allowing it to predict ratings even when overlapping interactions are rare.

**Weaknesses of SVD:**
- **Interpretability:** Latent factors are mathematical abstractions. We cannot easily explain to a user *why* a movie was recommended.
- **Training Time:** Retraining the matrix factorization when new data arrives is computationally expensive compared to simple similarity lookups.

**Comparison Against Item-Based CF:**
- SVD provides better overall predictive accuracy (RMSE) but acts as a "black box."
- Item-CF provides highly explainable recommendations (e.g., "Because you watched X"), which is often more valuable for user trust despite slightly higher RMSE.
